# Notes:
1. We had to recalculate the chain IDs (`full_scPDB.csv`) using the IFP.txt files in the scPDB dataset.

In [1]:
import pandas as pd
df = pd.read_csv('/home/skrhakv/CryptoBench/data/A-filter-ahojdb/all_apoholo_all_ligands/pairs.csv')

/tmp/slurm.214987/ipykernel_736781/1106849004.py:2: DtypeWarning: Columns (29,30,34,35,70,82,83,87,88) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/skrhakv/CryptoBench/data/A-filter-ahojdb/all_apoholo_all_ligands/pairs.csv')


In [2]:
import csv   
proteins = []
proteins_without_chains = []
scPDB_binding_sites = {}
with open('/home/skrhakv/cryptoshow-analysis/data/E-regular-binding-site-predictor/full_scPDB.csv', 'r') as f:
    csv_file = csv.reader(f, delimiter=';')
    for row in csv_file:
        proteins.append(f'{row[0]}{row[1]}')
        proteins_without_chains.append(row[0])
        scPDB_binding_sites[f'{row[0]}{row[1]}'] = row[3].split(' ')
scPDB_subset = df[df['apo_structure'].isin(proteins_without_chains) | df['holo_structure'].isin(proteins_without_chains)]

In [ ]:
def extract_residues(pocket_selection):
    residues_chunk = pocket_selection.split(' ')[-2][:-1]
    return residues_chunk.split('+')

binding_sites = {}
for protein_id in proteins:
    print(f'Processing {protein_id}...')
    binding_sites[protein_id] = set()
    pdb_id, chain_id = protein_id[:4], protein_id[4:]
    
    apo_subset = scPDB_subset[(scPDB_subset['apo_structure'] == pdb_id) & (scPDB_subset['apo_chains3'] == chain_id)]
    if not apo_subset.empty:
        for pocket_selection in apo_subset['apo_pocket_selection']:
            binding_sites[protein_id].update(extract_residues(pocket_selection))
    
    holo_subset = scPDB_subset[(scPDB_subset['holo_structure'] == pdb_id) & (scPDB_subset['holo_chains3'] == chain_id)]
    if not holo_subset.empty:
        for pocket_selection in holo_subset['holo_pocket_selection']:
            binding_sites[protein_id].update(extract_residues(pocket_selection))

import pickle
with open('/home/skrhakv/CryptoBench/data/A-filter-ahojdb/all_apoholo_all_ligands/binding_sites.pkl', 'wb') as f:
    pickle.dump(binding_sites, f)

Processing 1uh5B...
Processing 3bxoB...
Processing 4to3A...
Processing 2r2lB...
Processing 2ejuA...
Processing 2fm0A...
Processing 1sc6A...
Processing 1x8vA...
Processing 4kttA...
Processing 1s1uA...
Processing 2p6dA...
Processing 3s1cA...
Processing 4emtA...
Processing 3nhrB...
Processing 3um5A...
Processing 1mc1A...
Processing 1likA...
Processing 4rx6A...
Processing 4l2zB...
Processing 3kjsD...
Processing 1q4xA...
Processing 4ieiA...
Processing 3fmhA...
Processing 2j5xA...
Processing 3eonC...
Processing 1p7rA...
Processing 4eymA...
Processing 3bloA...
Processing 4j5jA...
Processing 4hsuA...
Processing 4y29A...
Processing 1meiA...
Processing 4h4sA...
Processing 3gb2A...
Processing 4i9zA...
Processing 2qa2A...
Processing 1e1qD...
Processing 3mw9A...
Processing 2aa0A...
Processing 3sr4A...
Processing 3sw1B...
Processing 5g53B...
Processing 3ia4D...
Processing 2o2uA...
Processing 1dwxB...
Processing 1pj2A...
Processing 4rxrA...
Processing 1z7eA...
Processing 1rx7A...
Processing 2j3kB...


# Merge
Here we are merging the scPDB and additional residues from AhojDB. We are safe because scPDB uses auth labels (see 1uh5 as an example), and AhojDB uses the same.

The resulting format of the CSV file is as follows (delimiter is ';'): 
```
PDB_ID;AUTH_CHAIN_ID;{CRYPTIC/NON_CRYPTIC};BINDING RESIDUES;SEQUENCE(NOT EXTRACTED YET)
```

In [ ]:
binding_sites_enhanced = {}
for protein_id in proteins:
    binding_sites_enhanced[protein_id] = binding_sites[protein_id].union(set(scPDB_binding_sites[protein_id]))

with open('/home/skrhakv/cryptoshow-analysis/data/E-regular-binding-site-predictor/scPDB_enhanced_binding_sites.csv', 'w') as f:
    for protein_id in proteins:
        f.write(f'{protein_id[:4]};{protein_id[4:]};NON_CRYPTIC;{" ".join(sorted(binding_sites_enhanced[protein_id]))};UNKNOWN\n')

## Extract the binding sites for reference
In the previous snippet we are merging all the binding sites together. Here let's extract each binding site alone, without merging.

In [4]:
def extract_residues(pocket_selection):
    residues_chunk = pocket_selection.split(' ')[-2][:-1]
    return residues_chunk.split('+')

binding_sites = {}
for protein_id in proteins:
    print(f'Processing {protein_id}...')
    binding_sites[protein_id] = []
    pdb_id, chain_id = protein_id[:4], protein_id[4:]
    
    apo_subset = scPDB_subset[(scPDB_subset['apo_structure'] == pdb_id) & (scPDB_subset['apo_chains3'] == chain_id)]
    if not apo_subset.empty:
        for pocket_selection, ligand in zip(apo_subset['apo_pocket_selection'], apo_subset['holo_query_ligs']):
            binding_sites[protein_id].append((extract_residues(pocket_selection), ligand))
    
    holo_subset = scPDB_subset[(scPDB_subset['holo_structure'] == pdb_id) & (scPDB_subset['holo_chains3'] == chain_id)]
    if not holo_subset.empty:
        for pocket_selection, ligand in zip(holo_subset['holo_pocket_selection'], holo_subset['holo_query_ligs']):
            binding_sites[protein_id].append((extract_residues(pocket_selection), ligand))


Processing 1uh5B...
Processing 3bxoB...
Processing 4to3A...
Processing 2r2lB...
Processing 2ejuA...
Processing 2fm0A...
Processing 1sc6A...
Processing 1x8vA...
Processing 4kttA...
Processing 1s1uA...
Processing 2p6dA...
Processing 3s1cA...
Processing 4emtA...
Processing 3nhrB...
Processing 3um5A...
Processing 1mc1A...
Processing 1likA...
Processing 4rx6A...
Processing 4l2zB...
Processing 3kjsD...
Processing 1q4xA...
Processing 4ieiA...
Processing 3fmhA...
Processing 2j5xA...
Processing 3eonC...
Processing 1p7rA...
Processing 4eymA...
Processing 3bloA...
Processing 4j5jA...
Processing 4hsuA...
Processing 4y29A...
Processing 1meiA...
Processing 4h4sA...
Processing 3gb2A...
Processing 4i9zA...
Processing 2qa2A...
Processing 1e1qD...
Processing 3mw9A...
Processing 2aa0A...
Processing 3sr4A...
Processing 3sw1B...
Processing 5g53B...
Processing 3ia4D...
Processing 2o2uA...
Processing 1dwxB...
Processing 1pj2A...
Processing 4rxrA...
Processing 1z7eA...
Processing 1rx7A...
Processing 2j3kB...


In [17]:

with open('/home/skrhakv/cryptoshow-analysis/data/E-regular-binding-site-predictor/scPDB-sites.csv', 'w') as f:
    for protein_id in proteins:
        for this_binding_site in binding_sites[protein_id]:
            f.write(f'{protein_id[:4]};{protein_id[4:]};{this_binding_site[1]};{" ".join(sorted(this_binding_site[0]))}\n')
